# Day 12 — The Standard PyTorch Training Loop

## 1. Learning Objectives
- Cement the 5 core steps of every PyTorch training loop.
- Combine `nn.Module`, `Loss`, and `Optimizer` together perfectly.
- Explain exactly *why* the order of operations matters.
- Build a muscle-memory template you will use for the rest of your career.

## 2. Prerequisites
- `torch.nn` (Day 9)
- Loss Functions (Day 10)
- Optimizers (Day 11)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

## 3. Concept Explanation
No matter if you are training a simple Linear Regression model on your laptop, or a 175-Billion parameter LLM on a supercomputer, the inner core of the training loop is *exactly the same 5 steps*.

1. **Forward Pass**: Pass data through the model to get predictions.
2. **Calculate Loss**: Compare predictions to true targets.
3. **Zero Gradients**: Clear out the old math from the previous step.
4. **Backward Pass**: Compute the new gradients (`loss.backward()`).
5. **Optimizer Step**: Update the weights.

## 8. Simple Example & 9. Code Walkthrough
Let's write out the "Holy Grail" PyTorch template.

In [ ]:
# --- 1. SETUP ---
torch.manual_seed(42)
X = torch.randn(100, 5)
y = torch.randn(100, 1)

model = nn.Linear(5, 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 5

# --- 2. THE LOOP ---
for epoch in range(epochs):
    
    # Step 1: Forward Pass
    # We feed X into the model. PyTorch builds the computational graph in the background.
    predictions = model(X)
    
    # Step 2: Calculate Loss
    # The criterion calculates how bad the predictions are compared to y.
    loss = criterion(predictions, y)
    
    # Step 3: Zero Gradients
    # We MUST clear the gradients from epoch N-1 before calculating them for epoch N.
    # Otherwise, they add up (accumulate), and the math breaks.
    optimizer.zero_grad()
    
    # Step 4: Backward Pass
    # Autograd traverses the graph backward, computing the derivative of the loss 
    # with respect to every weight in the model. Stores them in parameter.grad.
    loss.backward()
    
    # Step 5: Optimizer Step
    # The optimizer looks at the .grad of every parameter and adjusts the weights.
    optimizer.step()
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

## 11. Practice Exercise 1: Reorder the Loop
Can you swap Step 3 (`optimizer.zero_grad()`) and Step 4 (`loss.backward()`)? What about swapping Step 4 and Step 5?

Write down what you think would happen, then experiment.

**Solution:**
- Swapping `zero_grad()` and `loss.backward()`: Technically fine. You can zero gradients right before `backward()` or right after `step()`. Most people put it right before `backward()` to be safe.
- Swapping `loss.backward()` and `optimizer.step()`: **DISASTER**. If you step before backward, there are no new gradients! The optimizer will update weights using the *previous* epoch's gradients. Your model will fail to train.

## 13. Debugging Challenge
A student complains that their loss goes down for the first epoch, and then never changes again. What did they forget?

In [ ]:
buggy_model = nn.Linear(5, 1)
buggy_opt = optim.Adam(buggy_model.parameters(), lr=0.1)

for epoch in range(5):
    pred = buggy_model(X)
    loss = criterion(pred, y)
    loss.backward()
    # buggy_opt.step() # Uncomment to fix
    buggy_opt.zero_grad()
    print(loss.item())

**Solution:** They forgot `optimizer.step()`! The gradients are being calculated correctly (`loss.backward()`), and then erased (`zero_grad()`), but the weights are never actually updated.

## 17. Interview Questions
1. **Why does PyTorch force you to manually call `zero_grad()`? Why isn't it automatic?**
   *Answer*: Because gradient accumulation is a feature, not a bug. If you have a massive model that doesn't fit in GPU memory, you can run `backward()` on small chunks of data multiple times to accumulate the gradients, and then call `step()` once at the end. This allows training huge models on small GPUs.
2. **What happens if you forget `loss.backward()` but keep `optimizer.step()`?**
   *Answer*: The optimizer will try to look for `.grad` attributes on the parameters. If they don't exist (because backward wasn't called), it will crash or do nothing.

## 19. Day Summary
Commit this to muscle memory:
1. `outputs = model(inputs)`
2. `loss = criterion(outputs, targets)`
3. `optimizer.zero_grad()`
4. `loss.backward()`
5. `optimizer.step()`